# MIT 805 Part 1 — NYC HVFHV: Data Collection, Preparation & EDA

**Dataset:** NYC TLC High Volume For-Hire Vehicle (HVFHV) trip records — Uber, Lyft, Via, Juno.

This notebook produces every number and figure needed for the Part 1 report:

| Rubric criterion | Produced in |
|---|---|
| 1. Source, licence & suitability | Section 1 |
| 2. Dataset characteristics & scale (raw / working / processing) | Section 2, 4 |
| 3. Data preparation & quality | Sections 5–7 |
| 4. Exploratory Data Analysis | Sections 8–13 |
| 5. Seven Vs of Big Data | evidence from Sections 2–13 |
| 6. Value & limitations | Section 14 |

Aggregates are written to `output/` as CSV and figures to `figures/`. All heavy
computation is done in Spark; pandas is used only to plot small aggregated
results, which is what the brief requires.

**Runtime:** Colab CPU runtime is fine. Set `FAST_MODE = True` below if you are
short on time — it uses 7 months instead of 12 and still clears the 3 GB
processing-set floor.

## 0. Environment

In [ ]:
###!pip -q install pyspark==3.5.1 requests
###!java -version 2>&1 | head -1

In [ ]:
import os, json, csv, math
from datetime import date
from concurrent.futures import ThreadPoolExecutor

import requests
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

for d in ["data/raw", "output", "figures"]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
GB = 1024 ** 3
print("ready")

## 1. Source, licence and provenance

Everything here goes straight into the report's provenance paragraph. Do not
paraphrase it loosely — the licence claim is worth marks and must be accurate.

In [ ]:
PROVENANCE = {
    "publisher": "New York City Taxi and Limousine Commission (TLC)",
    "dataset": "High Volume For-Hire Vehicle (HVFHV) Trip Records",
    "landing_page": "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page",
    "data_dictionary": "https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_hvfhs.pdf",
    "user_guide": "https://www.nyc.gov/assets/tlc/downloads/pdf/trip_record_user_guide.pdf",
    "file_pattern": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_YYYY-MM.parquet",
    "mirror": "AWS Open Data Registry, s3://nyc-tlc (us-east-1)",
    "zone_lookup": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
    "zone_shapefile": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip",
    "licence": "NYC.gov Terms of Use (https://www.nyc.gov/home/terms-of-use.page); "
               "published as NYC Open Data. Free public reuse including research, "
               "with attribution to the City of New York.",
    "coverage_start": "2019-02",
    "publication_cadence": "Monthly, with roughly a two-month lag for vendor submissions",
    "accuracy_disclaimer": "TLC states the trip data was not created by the TLC and it "
                           "makes no representations as to accuracy; HVFHV records are "
                           "submitted by the licensed high-volume bases.",
    "pii": "No rider, driver, or vehicle identifiers. Locations are generalised to "
           "265 taxi zones; no GPS coordinates since July 2016.",
}
for k, v in PROVENANCE.items():
    print(f"{k:22s}: {v}")

with open("output/provenance.json", "w") as f:
    json.dump(PROVENANCE, f, indent=2)

## 2. Dataset tiers and size audit

The brief requires three reported sizes: **raw** (25–40 GB), **working**
(≥12 GB) and **processing** (≥3 GB).

The raw archive is measured with HTTP `HEAD` requests rather than downloaded —
we read `Content-Length` for every published monthly file. This gives an exact
byte count for the full public dataset in under a minute and is a defensible
method to describe in the report.

**Schema warning.** Files from Feb 2019 to late 2020 carry only a reduced column
set (licensee, base, pickup/dropoff time, PU/DO zone, `SR_Flag`). The rich
columns — `request_datetime`, `on_scene_datetime`, `trip_miles`, `driver_pay`,
WAV flags — appear from 2021, and `cbd_congestion_fee` was added from 2025 for
congestion pricing. The working set therefore starts in 2024 so the schema is
consistent, and this schema evolution is itself evidence for the *Variety* and
*Variability* Vs.

In [ ]:
FAST_MODE = False   # True -> 7 months instead of 12 for the processing tier

BASE = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def month_range(y0, m0, y1, m1):
    y, m = y0, m0
    while (y, m) <= (y1, m1):
        yield (y, m)
        m += 1
        if m == 13:
            y, m = y + 1, 1

def fname(y, m):  return f"fhvhv_tripdata_{y:04d}-{m:02d}.parquet"
def furl(y, m):   return f"{BASE}/{fname(y, m)}"

RAW        = list(month_range(2019, 2, 2026, 5))   # full published archive
WORKING    = list(month_range(2024, 1, 2026, 5))   # downloaded / preprocessed
PROCESSING = list(month_range(2025, 1, 2025, 7 if FAST_MODE else 12))

print(f"raw        : {len(RAW):3d} files  {RAW[0]} .. {RAW[-1]}")
print(f"working    : {len(WORKING):3d} files  {WORKING[0]} .. {WORKING[-1]}")
print(f"processing : {len(PROCESSING):3d} files  {PROCESSING[0]} .. {PROCESSING[-1]}")

In [ ]:
def head_size(ym):
    y, m = ym
    try:
        r = requests.head(furl(y, m), timeout=30, allow_redirects=True)
        return (f"{y}-{m:02d}", int(r.headers.get("Content-Length", 0))
                if r.status_code == 200 else 0)
    except Exception:
        return (f"{y}-{m:02d}", 0)

with ThreadPoolExecutor(max_workers=12) as pool:
    sizes = dict(pool.map(head_size, RAW))

missing = [k for k, v in sizes.items() if v == 0]
if missing:
    print("WARNING - no size returned for:", missing)

sizes_df = pd.DataFrame(
    [{"month": k, "bytes": v, "mb": v / 1024 / 1024} for k, v in sizes.items()]
).sort_values("month")
sizes_df.to_csv("output/file_sizes.csv", index=False)

def tier_gb(tier):
    return sum(sizes.get(f"{y}-{m:02d}", 0) for y, m in tier) / GB

tiers = pd.DataFrame([
    {"tier": "raw",        "n_files": len(RAW),
     "period": f"{RAW[0][0]}-{RAW[0][1]:02d} to {RAW[-1][0]}-{RAW[-1][1]:02d}",
     "gb_parquet": round(tier_gb(RAW), 2)},
    {"tier": "working",    "n_files": len(WORKING),
     "period": f"{WORKING[0][0]}-{WORKING[0][1]:02d} to {WORKING[-1][0]}-{WORKING[-1][1]:02d}",
     "gb_parquet": round(tier_gb(WORKING), 2)},
    {"tier": "processing", "n_files": len(PROCESSING),
     "period": f"{PROCESSING[0][0]}-{PROCESSING[0][1]:02d} to {PROCESSING[-1][0]}-{PROCESSING[-1][1]:02d}",
     "gb_parquet": round(tier_gb(PROCESSING), 2)},
])
tiers.to_csv("output/dataset_sizes.csv", index=False)
tiers

In [ ]:
# Monthly file size over time - shows both the growth of the service and the
# 2020 COVID collapse. Goes in the appendix.
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.bar(sizes_df["month"], sizes_df["mb"], width=0.85)
ax.set_ylabel("Parquet file size (MB)")
ax.set_title("HVFHV monthly file size, Feb 2019 - May 2026")
step = 6
ax.set_xticks(range(0, len(sizes_df), step))
ax.set_xticklabels(sizes_df["month"].iloc[::step], rotation=90)
fig.savefig("figures/fig_a1_monthly_file_size.png")
plt.close(fig)
print("saved figures/fig_a1_monthly_file_size.png")

## 3. Download the processing tier

Roughly 6 GB over 12 files (or 3.5 GB over 7 in fast mode). The working tier is
not downloaded in full here — its size is established by the HEAD audit above,
and the report should say so plainly rather than implying all 14 GB sat on disk.

If you want the full working tier on disk, change `TO_DOWNLOAD` to `WORKING`
and allow extra time; Colab has enough disk for it.

In [9]:
TO_DOWNLOAD = PROCESSING

def download(y, m):
    dest = f"data/raw/{fname(y, m)}"
    if os.path.exists(dest) and os.path.getsize(dest) == sizes.get(f"{y}-{m:02d}", -1):
        print(f"  have {fname(y, m)}")
        return
    print(f"  getting {fname(y, m)} ...", end="", flush=True)
    with requests.get(furl(y, m), stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(dest + ".part", "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
    os.replace(dest + ".part", dest)
    print(f" {os.path.getsize(dest)/1024/1024:.0f} MB")

for y, m in TO_DOWNLOAD:
    download(y, m)

# Zone lookup: 265 taxi zones with borough and service-zone labels.
if not os.path.exists("data/raw/taxi_zone_lookup.csv"):
    r = requests.get("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", timeout=60)
    open("data/raw/taxi_zone_lookup.csv", "wb").write(r.content)

on_disk = sum(os.path.getsize(f"data/raw/{f}") for f in os.listdir("data/raw"))
print(f"\non disk: {on_disk/GB:.2f} GB")

  getting fhvhv_tripdata_2025-01.parquet ... 468 MB
  getting fhvhv_tripdata_2025-02.parquet ... 440 MB
  getting fhvhv_tripdata_2025-03.parquet ... 479 MB
  getting fhvhv_tripdata_2025-04.parquet ... 465 MB
  getting fhvhv_tripdata_2025-05.parquet ... 493 MB
  getting fhvhv_tripdata_2025-06.parquet ... 468 MB
  getting fhvhv_tripdata_2025-07.parquet ... 465 MB
  getting fhvhv_tripdata_2025-08.parquet ... 457 MB
  getting fhvhv_tripdata_2025-09.parquet ... 459 MB
  getting fhvhv_tripdata_2025-10.parquet ... 494 MB
  getting fhvhv_tripdata_2025-11.parquet ... 484 MB
  getting fhvhv_tripdata_2025-12.parquet ... 511 MB

on disk: 5.55 GB


## 4. Start Spark and load

In [10]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("MIT805-HVFHV-Part1")
    .master("local[8]")
    .config("spark.driver.memory", "32g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.local.dir", "D:/805/tmp")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(spark.version, "| cores:", spark.sparkContext.defaultParallelism)

3.5.1 | cores: 8


In [11]:
paths = [f"data/raw/{fname(y, m)}" for y, m in PROCESSING]
df = spark.read.option("mergeSchema", "true").parquet(*paths)
df = df.cache()

n_rows = df.count()
n_cols = len(df.columns)
print(f"rows: {n_rows:,}   columns: {n_cols}")
df.printSchema()

rows: 243,589,684   columns: 25
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string 

In [15]:
# Uncompressed logical size. Parquet compresses this data roughly 5-8x, so the
# on-disk figure understates what Spark actually moves. Report both.
type_bytes = {"TimestampType": 8, "LongType": 8, "DoubleType": 8, "IntegerType": 4,
              "ShortType": 2, "ByteType": 1, "BooleanType": 1, "DateType": 4,
              "FloatType": 4}
est = 0
for f_ in df.schema.fields:
    tn = type(f_.dataType).__name__
    est += type_bytes.get(tn, 12)     # strings assumed ~12 bytes average
bytes_per_row = est
logical_gb = n_rows * bytes_per_row / GB

proc_parquet_gb = float(tiers.loc[tiers.tier == "processing", "gb_parquet"].iloc[0])
print(f"processing tier, Parquet on disk : {proc_parquet_gb:.2f} GB")
print(f"processing tier, uncompressed    : {logical_gb:.2f} GB "
      f"({bytes_per_row} bytes/row x {n_rows:,} rows)")
print(f"implied compression ratio        : {logical_gb/proc_parquet_gb:.1f}x")

with open("output/scale.json", "w") as f:
    json.dump({"rows": n_rows, "cols": n_cols,
               "bytes_per_row": bytes_per_row,
               "processing_parquet_gb": round(proc_parquet_gb, 2),
               "processing_uncompressed_gb": round(logical_gb, 2)}, f, indent=2)

processing tier, Parquet on disk : 5.55 GB
processing tier, uncompressed    : 54.45 GB (240 bytes/row x 243,589,684 rows)
implied compression ratio        : 9.8x


In [18]:
import sys
import setuptools._distutils as _distutils
import setuptools._distutils.version as _distutils_version

sys.modules["distutils"] = _distutils
sys.modules["distutils.version"] = _distutils_version

from distutils.version import LooseVersion
print("distutils shim installed:", LooseVersion("1.0.5"))

distutils shim installed: 1.0.5


## 5. Missing values

Criterion 3 wants *meaningful discussion*, not just a null count. Watch for
`originating_base_num` (frequently null — not every trip is dispatched through a
distinct originating base) and `on_scene_datetime` (null for some operators,
which breaks any wait-time analysis that assumes it is present).

In [19]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns
]).toPandas().T.rename(columns={0: "nulls"})
null_counts["pct"] = (null_counts["nulls"] / n_rows * 100).round(3)
null_counts = null_counts.sort_values("pct", ascending=False)
null_counts.to_csv("output/missingness.csv")
null_counts

,nulls,pct
originating_base_num,67385046,27.663
on_scene_datetime,10870353,4.463
hvfhs_license_num,0,0.000
dispatching_base_num,0,0.000
request_datetime,0,0.000
pickup_datetime,0,0.000
dropoff_datetime,0,0.000
PULocationID,0,0.000
DOLocationID,0,0.000
trip_miles,0,0.000


In [20]:
nz = null_counts[null_counts.pct > 0]
if len(nz):
    fig, ax = plt.subplots(figsize=(6, max(1.6, 0.28 * len(nz))))
    ax.barh(nz.index[::-1], nz["pct"][::-1])
    ax.set_xlabel("% of records null")
    ax.set_title("Missingness by column, HVFHV processing set")
    fig.savefig("figures/fig1_missingness.png")
    plt.close(fig)
    print("saved figures/fig1_missingness.png")
else:
    print("no nulls found - state this explicitly in the report")

saved figures/fig1_missingness.png


## 6. Duplicates

In [21]:
# There is no trip ID, so a duplicate must be defined on a composite key.
key = ["hvfhs_license_num", "pickup_datetime", "dropoff_datetime",
       "PULocationID", "DOLocationID", "base_passenger_fare", "driver_pay"]
n_distinct = df.select(*key).distinct().count()
dupes = n_rows - n_distinct
print(f"exact duplicates on composite key: {dupes:,} ({dupes/n_rows*100:.4f}%)")
print("NOTE: without a trip ID, some of these are genuine coincident trips, "
      "not data errors. Say so in the report rather than deleting them blindly.")

exact duplicates on composite key: 26 (0.0000%)
NOTE: without a trip ID, some of these are genuine coincident trips, not data errors. Say so in the report rather than deleting them blindly.


## 7. Validity audit

This is the section that separates a 3/3 from a 1/3 on data quality. We count
records that are internally inconsistent rather than merely extreme.

In [22]:
checks = {
    "trip_miles <= 0":            F.col("trip_miles") <= 0,
    "trip_time <= 0":             F.col("trip_time") <= 0,
    "trip_time > 6h":             F.col("trip_time") > 6 * 3600,
    "trip_miles > 100":           F.col("trip_miles") > 100,
    "base_passenger_fare < 0":    F.col("base_passenger_fare") < 0,
    "driver_pay <= 0":            F.col("driver_pay") <= 0,
    "dropoff <= pickup":          F.col("dropoff_datetime") <= F.col("pickup_datetime"),
    "pickup < request":           F.col("pickup_datetime") < F.col("request_datetime"),
    "unknown zone (264/265)":     F.col("PULocationID").isin(264, 265),
    "implied speed > 90 mph":     (F.col("trip_miles") / (F.col("trip_time") / 3600.0)) > 90,
}
rows = []
for name, cond in checks.items():
    c = df.filter(cond).count()
    rows.append({"check": name, "records": c, "pct": round(c / n_rows * 100, 4)})
quality = pd.DataFrame(rows).sort_values("records", ascending=False)
quality.to_csv("output/quality_checks.csv", index=False)
quality

,check,records,pct
7,pickup < request,2790424,1.1455
5,driver_pay <= 0,58781,0.0241
0,trip_miles <= 0,28128,0.0115
3,trip_miles > 100,22397,0.0092
4,base_passenger_fare < 0,13495,0.0055
6,dropoff <= pickup,12231,0.0050
8,unknown zone (264/265),10398,0.0043
2,trip_time > 6h,465,0.0002
9,implied speed > 90 mph,94,0.0000
1,trip_time <= 0,31,0.0000


In [23]:
# Cleaned frame used for the remaining EDA. Keep the rule set explicit and
# reproduce this exact list in the report's preparation paragraph.
clean = df.filter(
    (F.col("trip_miles") > 0) &
    (F.col("trip_time") > 0) & (F.col("trip_time") <= 6 * 3600) &
    (F.col("base_passenger_fare") >= 0) &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    (~F.col("PULocationID").isin(264, 265)) &
    (~F.col("DOLocationID").isin(264, 265))
).cache()

n_clean = clean.count()
print(f"retained {n_clean:,} of {n_rows:,} records "
      f"({n_clean/n_rows*100:.2f}%) - dropped {n_rows-n_clean:,}")

retained 232,463,392 of 243,589,684 records (95.43%) - dropped 11,126,292


In [27]:
###something here is removing ~11 million records that the audit never flagged. It's almost certainly DOLocationID in (264, 265). The audit only checked pickup zones (10,398). Dropoff unknowns are a different story: zone 265 is "Outside of NYC" and 264 is unknown, and a lot of trips legitimately end in New Jersey, Westchester or at Newark.
(df.groupBy(F.col("DOLocationID").isin(264, 265).alias("do_unknown"),
            F.col("PULocationID").isin(264, 265).alias("pu_unknown"))
   .agg(F.count("*").alias("trips"),
        F.avg("trip_miles").alias("avg_miles"),
        F.avg("base_passenger_fare").alias("avg_fare"))
   .show())

+----------+----------+---------+------------------+------------------+
|do_unknown|pu_unknown|    trips|         avg_miles|          avg_fare|
+----------+----------+---------+------------------+------------------+
|      true|      true|     3840| 5.411554687499999|24.343151041666662|
|     false|     false|232514993|4.4759526490103445|24.895373030708104|
|      true|     false| 11064293|16.735033371675872| 70.07914309933774|
|     false|      true|     6558| 6.596792924672156|25.745510826471477|
+----------+----------+---------+------------------+------------------+



## 8. Distributional summary

In [28]:
num_cols = ["trip_miles", "trip_time", "base_passenger_fare", "tolls",
            "sales_tax", "congestion_surcharge", "airport_fee", "tips", "driver_pay"]
num_cols = [c for c in num_cols if c in clean.columns]

desc = clean.select(num_cols).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).toPandas().set_index("summary").T
desc.to_csv("output/numeric_summary.csv")
desc

summary,count,mean,stddev,min,25%,50%,75%,max
trip_miles,232463392,4.47637087815518,4.430769655388438,0.001,1.49,2.823,5.82,332.21
trip_time,232463392,1141.1125219621676,787.0502358635479,1,582,941,1479,21552
base_passenger_fare,232463392,24.897286954917583,19.639906526451202,0.0,12.17,18.85,30.52,1679.51
tolls,232463392,0.7260360502240868,2.5713514166191858,0.0,0.0,0.0,0.0,107.62
sales_tax,232463392,2.208534117999939,1.818896058837774,0.0,1.04,1.66,2.72,151.36
congestion_surcharge,232463392,1.0219271976810869,1.3243036708958187,0.0,0.0,0.0,2.75,8.25
airport_fee,232463392,0.20855471406869946,0.6952490431976915,0.0,0.0,0.0,0.0,10.0
tips,232463392,1.073877539780862,3.0693479838502196,0.0,0.0,0.0,0.0,178.12
driver_pay,232463392,19.25845513172106,14.67364288037217,-55.02,8.94,14.97,24.93,844.37


In [29]:
(clean.filter(F.col("driver_pay") < 0)
   .agg(F.count("*").alias("n"),
        F.min("driver_pay").alias("min_pay"),
        F.avg("driver_pay").alias("avg_pay"),
        F.avg("base_passenger_fare").alias("avg_fare")).show())

+---+-------+------------------+-----------------+
|  n|min_pay|           avg_pay|         avg_fare|
+---+-------+------------------+-----------------+
|531| -55.02|-8.233596986817325|76.61873822975518|
+---+-------+------------------+-----------------+



In [30]:
# Histograms from Spark-side binning - no toPandas() on the full frame.
def spark_hist(colname, lo, hi, bins=60):
    w = (hi - lo) / bins
    h = (clean.filter((F.col(colname) >= lo) & (F.col(colname) < hi))
              .select(F.floor((F.col(colname) - lo) / w).alias("b"))
              .groupBy("b").count().orderBy("b").toPandas())
    h["edge"] = lo + h["b"] * w
    return h

fig, axes = plt.subplots(1, 3, figsize=(11, 2.9))
for ax, (c, lo, hi, lab) in zip(axes, [
        ("trip_miles", 0, 25, "Trip distance (miles)"),
        ("trip_time", 0, 4800, "Trip duration (seconds)"),
        ("base_passenger_fare", 0, 100, "Base passenger fare (USD)")]):
    h = spark_hist(c, lo, hi)
    ax.bar(h["edge"], h["count"], width=(hi - lo) / 60, align="edge")
    ax.set_xlabel(lab); ax.set_ylabel("trips")
fig.suptitle("Distribution of core trip attributes", y=1.04)
fig.savefig("figures/fig2_distributions.png")
plt.close(fig)
print("saved figures/fig2_distributions.png")

saved figures/fig2_distributions.png


## 9. Market structure by licensee

In [31]:
LICENSEES = {"HV0002": "Juno", "HV0003": "Uber", "HV0004": "Via", "HV0005": "Lyft"}
lic = (clean.groupBy("hvfhs_license_num")
            .agg(F.count("*").alias("trips"),
                 F.avg("trip_miles").alias("avg_miles"),
                 F.avg("base_passenger_fare").alias("avg_fare"),
                 F.avg("driver_pay").alias("avg_driver_pay"))
            .toPandas())
lic["operator"] = lic["hvfhs_license_num"].map(LICENSEES).fillna("unknown")
lic["share_pct"] = (lic["trips"] / lic["trips"].sum() * 100).round(2)
lic = lic.sort_values("trips", ascending=False)
lic.to_csv("output/market_share.csv", index=False)
lic[["operator", "trips", "share_pct", "avg_miles", "avg_fare", "avg_driver_pay"]]

,operator,trips,share_pct,avg_miles,avg_fare,avg_driver_pay
1,Uber,167756559,72.16,4.536869,25.472332,19.550590
0,Lyft,64706833,27.84,4.319526,23.406447,18.501078


## 10. Temporal structure

In [32]:
daily = (clean.groupBy(F.to_date("pickup_datetime").alias("day"))
              .count().orderBy("day").toPandas())
daily.to_csv("output/daily_trips.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(pd.to_datetime(daily["day"]), daily["count"], lw=0.8)
ax.set_ylabel("trips per day"); ax.set_title("Daily HVFHV trip volume")
fig.savefig("figures/fig3_daily_volume.png")
plt.close(fig)

print(f"mean {daily['count'].mean():,.0f} trips/day   "
      f"min {daily['count'].min():,}   max {daily['count'].max():,}")
print(f"implied mean arrival rate: {daily['count'].mean()/86400:,.1f} trips/second")


mean 636,886 trips/day   min 495,548   max 887,777
implied mean arrival rate: 7.4 trips/second


In [33]:
print(daily.nlargest(5, "count"))
print(daily.nsmallest(5, "count"))

            day   count
346  2025-12-13  887777
304  2025-11-01  882305
345  2025-12-12  845064
352  2025-12-19  835871
339  2025-12-06  833599
            day   count
265  2025-09-23  495548
229  2025-08-18  497161
145  2025-05-26  498048
243  2025-09-01  501250
358  2025-12-25  501610


In [34]:
hw = (clean.groupBy(F.dayofweek("pickup_datetime").alias("dow"),
                    F.hour("pickup_datetime").alias("hour"))
           .count().toPandas())
piv = hw.pivot(index="dow", columns="hour", values="count").fillna(0)
piv.index = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

fig, ax = plt.subplots(figsize=(9, 2.4))
im = ax.imshow(piv.values, aspect="auto", cmap="magma")
ax.set_yticks(range(7)); ax.set_yticklabels(piv.index)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("hour of day")
ax.set_title("Trip starts by hour and day of week")
ax.grid(False)
fig.colorbar(im, ax=ax, label="trips")
fig.savefig("figures/fig4_hour_dow_heatmap.png")
plt.close(fig)
print("saved figures/fig4_hour_dow_heatmap.png")

saved figures/fig4_hour_dow_heatmap.png


In [35]:
top = piv.stack().sort_values(ascending=False).head(5)
bot = piv.stack().sort_values().head(3)
print("busiest hour-of-week cells:\n", top, "\n")
print("quietest:\n", bot, "\n")
print("weekday 8am:", int(piv.loc["Wed", 8]), " weekday 6pm:", int(piv.loc["Wed", 18]))
print("Sat 01:00:", int(piv.loc["Sat", 1]), " Sun 01:00:", int(piv.loc["Sun", 1]))

busiest hour-of-week cells:
      hour
Sat  19      2283178
     22      2255931
Fri  19      2220931
Sat  23      2218001
     18      2209694
dtype: int64 

quietest:
      hour
Tue  3       245562
     2       274355
Mon  3       308997
dtype: int64 

weekday 8am: 2079169  weekday 6pm: 2015243
Sat 01:00: 1400889  Sun 01:00: 1599989


## 11. Service quality: wait time, shared rides, accessibility

`request_datetime` → `pickup_datetime` gives passenger wait, and
`on_scene_datetime` separates driver travel from passenger boarding. The
request/match flag pairs are the strongest material here: they record demand
that was *asked for and not delivered*, which is unusual in an open dataset and
makes an excellent central question for Part 2.

In [36]:
if "request_datetime" in clean.columns:
    waits = clean.withColumn(
        "wait_s",
        F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
    ).filter((F.col("wait_s") >= 0) & (F.col("wait_s") <= 3600))

    q = waits.approxQuantile("wait_s", [0.1, 0.25, 0.5, 0.75, 0.9, 0.95], 0.01)
    print("wait time (s) deciles p10/p25/p50/p75/p90/p95:",
          [round(x) for x in q])
    print(f"median wait: {q[2]/60:.1f} min    p90: {q[4]/60:.1f} min")

    by_hour = (waits.groupBy(F.hour("request_datetime").alias("hour"))
                    .agg(F.avg("wait_s").alias("mean_wait_s"),
                         F.count("*").alias("trips"))
                    .orderBy("hour").toPandas())
    by_hour.to_csv("output/wait_by_hour.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 2.6))
    ax.plot(by_hour["hour"], by_hour["mean_wait_s"] / 60, marker="o", ms=3)
    ax.set_xlabel("hour of day"); ax.set_ylabel("mean wait (min)")
    ax.set_title("Mean passenger wait by hour")
    fig.savefig("figures/fig5_wait_by_hour.png")
    plt.close(fig)

wait time (s) deciles p10/p25/p50/p75/p90/p95: [118, 168, 241, 347, 497, 627]
median wait: 4.0 min    p90: 8.3 min


In [38]:
(waits.groupBy("hvfhs_license_num")
      .agg(F.count("*").alias("trips"),
           F.avg("wait_s").alias("mean_wait_s"),
           F.expr("percentile_approx(wait_s, 0.5)").alias("median_wait_s"),
           F.expr("percentile_approx(wait_s, 0.9)").alias("p90_wait_s"))
      .show())

+-----------------+---------+-----------------+-------------+----------+
|hvfhs_license_num|    trips|      mean_wait_s|median_wait_s|p90_wait_s|
+-----------------+---------+-----------------+-------------+----------+
|           HV0005| 63619758|291.1774997163617|          253|       495|
|           HV0003|166242482| 283.970749281763|          237|       505|
+-----------------+---------+-----------------+-------------+----------+



In [39]:
flag_pairs = [("shared_request_flag", "shared_match_flag", "Shared ride"),
              ("wav_request_flag", "wav_match_flag", "Wheelchair-accessible (WAV)")]
rows = []
for req, mat, label in flag_pairs:
    if req in clean.columns and mat in clean.columns:
        r = (clean.groupBy(req, mat).count().toPandas())
        r.columns = ["requested", "matched", "trips"]
        req_yes = r[r.requested == "Y"]["trips"].sum()
        matched = r[(r.requested == "Y") & (r.matched == "Y")]["trips"].sum()
        rate = matched / req_yes * 100 if req_yes else float("nan")
        rows.append({"service": label, "requested": int(req_yes),
                     "matched": int(matched), "match_rate_pct": round(rate, 2),
                     "request_rate_pct": round(req_yes / n_clean * 100, 3)})
match = pd.DataFrame(rows)
match.to_csv("output/match_rates.csv", index=False)
match

,service,requested,matched,match_rate_pct,request_rate_pct
0,Shared ride,6889088,3939396,57.18,2.964
1,Wheelchair-accessible (WAV),685364,685353,100.00,0.295


## 12. Geography — joining the taxi zone lookup

In [40]:
zones = (spark.read.option("header", True).csv("data/raw/taxi_zone_lookup.csv")
              .withColumn("LocationID", F.col("LocationID").cast("int")))
zones.show(3, truncate=False)

top_pu = (clean.groupBy("PULocationID")
               .agg(F.count("*").alias("trips"),
                    F.avg("base_passenger_fare").alias("avg_fare"))
               .join(zones, F.col("PULocationID") == F.col("LocationID"))
               .select("Zone", "Borough", "trips", "avg_fare")
               .orderBy(F.desc("trips")).limit(15).toPandas())
top_pu.to_csv("output/top_pickup_zones.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.barh(top_pu["Zone"][::-1], top_pu["trips"][::-1])
ax.set_xlabel("trips"); ax.set_title("Top 15 pickup zones")
fig.savefig("figures/fig6_top_pickup_zones.png")
plt.close(fig)
top_pu

+----------+-------+-----------------------+------------+
|LocationID|Borough|Zone                   |service_zone|
+----------+-------+-----------------------+------------+
|1         |EWR    |Newark Airport         |EWR         |
|2         |Queens |Jamaica Bay            |Boro Zone   |
|3         |Bronx  |Allerton/Pelham Gardens|Boro Zone   |
+----------+-------+-----------------------+------------+
only showing top 3 rows



,Zone,Borough,trips,avg_fare
0,LaGuardia Airport,Queens,4250005,51.709302
1,JFK Airport,Queens,3247011,62.517201
2,Crown Heights North,Brooklyn,3138991,20.769513
3,East Village,Manhattan,2768594,26.395837
4,East New York,Brooklyn,2691348,18.008248
5,Bushwick South,Brooklyn,2658433,21.975880
6,Times Sq/Theatre District,Manhattan,2605542,38.426761
7,Midtown Center,Manhattan,2575190,36.353041
8,TriBeCa/Civic Center,Manhattan,2506574,32.099295
9,Williamsburg (North Side),Brooklyn,2401220,26.369987


In [41]:
pu = zones.select(F.col("LocationID").alias("pu_id"), F.col("Borough").alias("pu_boro"))
do = zones.select(F.col("LocationID").alias("do_id"), F.col("Borough").alias("do_boro"))
flows = (clean.join(pu, clean.PULocationID == pu.pu_id)
              .join(do, clean.DOLocationID == do.do_id)
              .groupBy("pu_boro", "do_boro").count().toPandas())
mat = flows.pivot(index="pu_boro", columns="do_boro", values="count").fillna(0)
mat.to_csv("output/borough_flows.csv")
(mat / mat.values.sum() * 100).round(2)

do_boro,Bronx,Brooklyn,EWR,Manhattan,Queens,Staten Island
pu_boro,,,,,,
Bronx,9.88,0.17,0.01,2.16,0.49,0.00
Brooklyn,0.17,21.31,0.09,3.09,3.01,0.10
EWR,0.00,0.00,0.00,0.00,0.00,0.00
Manhattan,2.26,3.50,0.61,26.18,4.01,0.05
Queens,0.46,2.69,0.02,3.25,14.84,0.02
Staten Island,0.00,0.12,0.02,0.04,0.02,1.40


## 13. Economics — driver share of the fare

`driver_pay` against `base_passenger_fare` gives an observable take rate. This
is directly relevant to TLC minimum-pay rulemaking and is one of the clearest
"Value" arguments available in this dataset.

In [42]:
econ = (clean.filter(F.col("base_passenger_fare") > 0)
             .groupBy(F.date_trunc("month", "pickup_datetime").alias("month"))
             .agg(F.sum("base_passenger_fare").alias("fares"),
                  F.sum("driver_pay").alias("driver_pay"),
                  F.sum("tips").alias("tips"),
                  F.count("*").alias("trips"))
             .orderBy("month").toPandas())
econ["driver_share_pct"] = (econ["driver_pay"] / econ["fares"] * 100).round(2)
econ["tip_rate_pct"] = (econ["tips"] / econ["fares"] * 100).round(2)
econ.to_csv("output/monthly_economics.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.plot(econ["month"], econ["driver_share_pct"], marker="o", ms=3, label="driver pay")
ax.plot(econ["month"], econ["tip_rate_pct"], marker="s", ms=3, label="tips")
ax.set_ylabel("% of base passenger fare"); ax.legend()
ax.set_title("Driver pay and tips as a share of fares")
fig.autofmt_xdate()
fig.savefig("figures/fig7_driver_share.png")
plt.close(fig)
econ[["month", "trips", "driver_share_pct", "tip_rate_pct"]]

,month,trips,driver_share_pct,tip_rate_pct
0,2025-01-01,19571000,76.70,4.19
1,2025-02-01,18530370,76.23,3.95
2,2025-03-01,19565537,73.28,3.92
3,2025-04-01,18862171,76.90,4.13
4,2025-05-01,20113794,77.49,4.28
5,2025-06-01,18922431,77.32,4.29
6,2025-07-01,18710418,78.19,4.30
7,2025-08-01,18170019,77.99,4.21
8,2025-09-01,18479350,76.93,4.40
9,2025-10-01,20285398,79.48,4.77


In [43]:
# Congestion pricing: cbd_congestion_fee exists only from 2025. Coverage and
# magnitude here are strong evidence for the Variability V.
if "cbd_congestion_fee" in clean.columns:
    cbd = (clean.agg(
        F.count(F.when(F.col("cbd_congestion_fee") > 0, 1)).alias("charged"),
        F.avg(F.when(F.col("cbd_congestion_fee") > 0,
                     F.col("cbd_congestion_fee"))).alias("avg_fee_when_charged"),
        F.sum("cbd_congestion_fee").alias("total_fee")).toPandas())
    cbd["pct_of_trips_charged"] = (cbd["charged"] / n_clean * 100).round(2)
    cbd.to_csv("output/congestion_fee.csv", index=False)
    display(cbd)
else:
    print("cbd_congestion_fee not present - processing tier predates 2025")

,charged,avg_fee_when_charged,total_fee,pct_of_trips_charged
0,78083012,1.500002,117124705.5,33.59


## 14. Numbers for the report

Everything below is what you paste into the LaTeX skeleton. Check each figure
before you use it — do not report a number this notebook did not actually
produce.

In [44]:
summary = {
    "raw_gb_parquet": float(tiers.loc[tiers.tier == "raw", "gb_parquet"].iloc[0]),
    "raw_files": int(tiers.loc[tiers.tier == "raw", "n_files"].iloc[0]),
    "working_gb_parquet": float(tiers.loc[tiers.tier == "working", "gb_parquet"].iloc[0]),
    "processing_gb_parquet": float(tiers.loc[tiers.tier == "processing", "gb_parquet"].iloc[0]),
    "processing_gb_uncompressed": round(logical_gb, 2),
    "processing_rows": n_rows,
    "processing_cols": n_cols,
    "rows_after_cleaning": n_clean,
    "pct_retained": round(n_clean / n_rows * 100, 2),
    "mean_trips_per_day": int(daily["count"].mean()),
    "peak_trips_per_day": int(daily["count"].max()),
    "trips_per_second_mean": round(daily["count"].mean() / 86400, 1),
}
with open("output/report_numbers.json", "w") as f:
    json.dump(summary, f, indent=2)
for k, v in summary.items():
    print(f"{k:32s}: {v}")

raw_gb_parquet                  : 37.11
raw_files                       : 88
working_gb_parquet              : 13.32
processing_gb_parquet           : 5.55
processing_gb_uncompressed      : 54.45
processing_rows                 : 243589684
processing_cols                 : 25
rows_after_cleaning             : 232463392
pct_retained                    : 95.43
mean_trips_per_day              : 636886
peak_trips_per_day              : 887777
trips_per_second_mean           : 7.4


In [46]:
econ_full = econ.copy()
total_fares = econ_full["fares"].sum()
total_driver = econ_full["driver_pay"].sum()
jan_rate = econ_full.iloc[0]["driver_share_pct"] / 100
actual_share = total_driver / total_fares

print(f"total base fares      : ${total_fares:,.0f}")
print(f"total driver pay      : ${total_driver:,.0f}")
print(f"annual driver share   : {actual_share*100:.2f}%")
print(f"if held at Jan rate   : ${total_fares*jan_rate:,.0f}")
print(f"difference            : ${total_driver - total_fares*jan_rate:,.0f}")
print(econ_full[["month","trips","driver_share_pct","tip_rate_pct"]].to_string())

total base fares      : $5,787,707,777
total driver pay      : $4,470,590,851
annual driver share   : 77.24%
if held at Jan rate   : $4,439,171,865
difference            : $31,418,986
        month     trips  driver_share_pct  tip_rate_pct
0  2025-01-01  19571000             76.70          4.19
1  2025-02-01  18530370             76.23          3.95
2  2025-03-01  19565537             73.28          3.92
3  2025-04-01  18862171             76.90          4.13
4  2025-05-01  20113794             77.49          4.28
5  2025-06-01  18922431             77.32          4.29
6  2025-07-01  18710418             78.19          4.30
7  2025-08-01  18170019             77.99          4.21
8  2025-09-01  18479350             76.93          4.40
9  2025-10-01  20285398             79.48          4.77
10 2025-11-01  19834070             78.70          4.47
11 2025-12-01  21067990             77.58          4.73


In [50]:
import os
print("cwd:", os.getcwd())
print("data/raw exists:", os.path.exists("data/raw"))
print("../data/raw exists:", os.path.exists("../data/raw"))

cwd: D:\805\notebooks
data/raw exists: False
../data/raw exists: True


In [51]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working directory:", os.getcwd())

working directory: D:\805


In [52]:
import requests, os

RAW = r"D:\805\data\raw"
os.makedirs(RAW, exist_ok=True)

u   = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2019-06.parquet"
p   = os.path.join(RAW, "fhvhv_tripdata_2019-06.parquet")
p25 = os.path.join(RAW, "fhvhv_tripdata_2025-06.parquet")

if not os.path.exists(p):
    print("downloading 2019-06 ...", end="", flush=True)
    with requests.get(u, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(p + ".part", "wb") as f:
            for c in r.iter_content(1024 * 1024):
                f.write(c)
    os.replace(p + ".part", p)
    print(f" {os.path.getsize(p)/1024/1024:.0f} MB")

# forward slashes for Spark's Hadoop path parser
d19 = spark.read.parquet(p.replace("\\", "/"));   n19 = d19.count()
d25 = spark.read.parquet(p25.replace("\\", "/")); n25 = d25.count()

mb19, mb25 = os.path.getsize(p)/1024/1024, os.path.getsize(p25)/1024/1024
print(f"\n2019-06: {n19:,} rows, {len(d19.columns)} cols, {mb19:.0f} MB")
print(f"2025-06: {n25:,} rows, {len(d25.columns)} cols, {mb25:.0f} MB")
print(f"\n2025 is {n25/n19*100:.1f}% of 2019 volume")
print(f"bytes/row: 2019 {os.path.getsize(p)/n19:.1f}   2025 {os.path.getsize(p25)/n25:.1f}")
print(f"\n2019 columns: {d19.columns}")

downloading 2019-06 ... 511 MB

2019-06: 21,001,990 rows, 24 cols, 511 MB
2025-06: 19,868,009 rows, 25 cols, 468 MB

2025 is 94.6% of 2019 volume
bytes/row: 2019 25.5   2025 24.7

2019 columns: ['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']


In [47]:
# --- Check : verify arithmetic claims made in the report ---------------------
uber_pct, lyft_pct = 72.16, 27.84
hhi = uber_pct**2 + lyft_pct**2
print(f"HHI                        : {hhi:,.0f}   (report says ~5980)")

n_clean_rows = 232463392
mean_fare = 24.897286954917583
annual_fares = n_clean_rows * mean_fare
print(f"annual base fares (derived): ${annual_fares:,.0f}")
print(f"annual base fares (summed) : ${econ['fares'].sum():,.0f}   <- use this one")
print(f"one point of share is worth: ${econ['fares'].sum()/100:,.0f}")

print(f"\nrode alone                 : {6889088-3939396:,}   (report says 2.95m)")
print(f"shared match rate          : {3939396/6889088*100:.2f}%")
print(f"WAV request rate           : {685364/n_clean_rows*100:.3f}%")

print(f"\nhour-of-week ratio         : {2283178/245562:.1f}x  (report says 9.3)")
print(f"day ratio                  : {887777/495548:.2f}x")
print(f"Wed 08:00 vs 18:00         : {2079169/2015243:.3f}x")

print(f"\nCBD fee, Crown Heights     : {1.50/20.77*100:.1f}%   (report says 7.2)")
print(f"CBD fee, JFK               : {1.50/62.52*100:.1f}%   (report says 2.4)")
print(f"CBD trips charged          : {78083012/n_clean_rows*100:.2f}%")

print(f"\nintra-borough total        : {26.18+21.31+14.84+9.88:.2f}%  (report says 72.2)")
print(f"compression ratio          : {54.45/5.55:.2f}x  (report says 9.81)")
print(f"Uber:Lyft volume           : {167756559/64706833:.2f}:1")
print(f"wait gap, median           : {253-237}s  ({(253-237)/237*100:.1f}%)")

HHI                        : 5,982   (report says ~5980)
annual base fares (derived): $5,787,707,777
annual base fares (summed) : $5,787,707,777   <- use this one
one point of share is worth: $57,877,078

rode alone                 : 2,949,692   (report says 2.95m)
shared match rate          : 57.18%
WAV request rate           : 0.295%

hour-of-week ratio         : 9.3x  (report says 9.3)
day ratio                  : 1.79x
Wed 08:00 vs 18:00         : 1.032x

CBD fee, Crown Heights     : 7.2%   (report says 7.2)
CBD fee, JFK               : 2.4%   (report says 2.4)
CBD trips charged          : 33.59%

intra-borough total        : 72.21%  (report says 72.2)
compression ratio          : 9.81x  (report says 9.81)
Uber:Lyft volume           : 2.59:1
wait gap, median           : 16s  (6.8%)


In [45]:
print("output/ :", sorted(os.listdir("output")))
print("figures/:", sorted(os.listdir("figures")))

output/ : ['borough_flows.csv', 'congestion_fee.csv', 'daily_trips.csv', 'dataset_sizes.csv', 'file_sizes.csv', 'market_share.csv', 'match_rates.csv', 'missingness.csv', 'monthly_economics.csv', 'numeric_summary.csv', 'provenance.json', 'quality_checks.csv', 'report_numbers.json', 'scale.json', 'top_pickup_zones.csv', 'wait_by_hour.csv']
figures/: ['fig1_missingness.png', 'fig2_distributions.png', 'fig3_daily_volume.png', 'fig4_hour_dow_heatmap.png', 'fig5_wait_by_hour.png', 'fig6_top_pickup_zones.png', 'fig7_driver_share.png', 'fig_a1_monthly_file_size.png']


### Zipping results back out of Colab

```python
!zip -qr mit805_part1_outputs.zip output figures
from google.colab import files; files.download("mit805_part1_outputs.zip")
```

Commit `output/` and `figures/` to the repo. Do **not** commit `data/raw/` — the
TLC files are large and the brief explicitly asks you to reference the source
rather than redistribute it. `.gitignore` already excludes it.